# 문항 1 : 아실 아파트 목록, 매물 크롤링

## 요구사항 1. 아실 사이트에서 수집을 두 단계로 분리해 작성하시오.

대상: https://asil.kr/asil/index.jsp

1단계 collect_apt.py — 행정동 별로 아파트 목록을 모아 apts.csv로 저장

2단계 collect_forsale.py — apts.csv를 읽어 매물 페이지를 수집해 forsales.csv로 저장 (status를 최신화, done, failed)

apts.csv는 다음 컬럼을 가질 것: seq, status, collected_at +a (상세칼럼들)

status의 초기값은 pending

요청 간 0.5초 이상 지연

In [3]:
import time
import csv
from datetime import datetime
import requests

In [ ]:
def collect_apt():
    dong_codes = ["1117013100"] 
    
    all_buildings = []
    
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Referer": "https://asil.kr/app/apt_list.jsp"  
    }
    
    print(f"총 {len(dong_codes)}개 행정동 데이터 수집을 시작합니다.")

    for idx, dong_code in enumerate(dong_codes):
        url = "https://asil.kr/app/data/data_apt_list.jsp?"
        params = {
            "dong": "1117013100",
            "building" : "",
            "household": "50",
            "order": "0",
            "order_type": "0"
        }
        
        print(f"[{idx+1}/{len(dong_codes)}] 행정동 코드 {dong_code} 요청 중...")
        
        try:
            response = requests.get(url, headers=headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data_list = response.json()
                current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                
                for item in data_list:
                    building_info = {
                        "seq": item.get("seq", ""),
                        "status": "PENDING",
                        "수집시각": current_time,
                        "타입": item.get("building", ""),
                        "아파트명": item.get("name", ""),
                        "동": item.get("dongname", ""),
                        "번지": item.get("bungi", ""),
                        "준공연도": item.get("movein", ""),
                        "세대수": item.get("household", ""),
                        "매물수": item.get("offer".replace("매물","0"), ""),
                    }
                    all_buildings.append(building_info)
                
                print(f"성공: {len(data_list)}개의 건물 데이터 가져옴.")
            else:
                print(f"요청 실패 (상태 코드: {response.status_code})")
                
        except Exception as e:
            print(f"에러 발생: {e}")


        time.sleep(0.6)   

        return all_buildings 

In [33]:
if __name__ == "__main__":
    all_buildings = collect_apt()

if all_buildings:
        csv_file = "apts.csv"
        fieldnames = list(all_buildings[0].keys())
        
        try:
            with open(csv_file, "w", encoding="utf-8-sig", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()  
                writer.writerows(all_buildings) 
                
            print(f"수집 완료! 총 {len(all_buildings)}개의 데이터가 '{csv_file}'에 저장되었습니다.")
        except Exception as e:
            print(f"파일 저장 중 에러 발생: {e}")
else:
    print("수집된 데이터가 없어 파일을 생성하지 않았습니다.")

총 1개 행정동 데이터 수집을 시작합니다.
[1/1] 행정동 코드 1117013100 요청 중...
성공: 20개의 건물 데이터 가져옴.
수집 완료! 총 20개의 데이터가 'apts.csv'에 저장되었습니다.


In [7]:
headers_info = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Referer": "https://asil.kr/app/apt_info.jsp?os=pc&move=no&building={building}&apt={seq}&min_py=0&max_py=70"  
    }

headers_sale = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Referer": "https://asil.kr/"  
    }

def collect_forsale():
    INFO_URL = "https://asil.kr/app/data/data_apt_dong.jsp?"                
    SALE_URL = "https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx"
    
    apts_file = "apts.csv"
    
    # 1. 기존 apts.csv 파일 읽기 및 대상 필터링
    apts_data = []
    try:
        with open(apts_file, mode="r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            apts_data = list(reader)
    except FileNotFoundError:
        print(f"'{apts_file}' 파일이 존재하지 않습니다. 1단계를 먼저 실행해 주세요.")
        return

    
    target_apts = []
    for apt in apts_data:
        offer_val = apt.get("매물수", "0").strip()
        if not offer_val or offer_val == "0" or offer_val == "":
            apt["status"] = "DONE"
        if offer_val and offer_val != "0" and apt.get("status") == "PENDING":
            target_apts.append(apt)

    print(f"총 {len(apts_data)}개 중 매물이 존재하는 {len(target_apts)}개의 아파트를 대상으로 수집을 진행합니다.\n")

    if not target_apts:
        print("수집할 대상(PENDING 상태 및 매물 보유)이 없습니다.")
        return

    all_forsales = []

    # 2. 필터링된 아파트 단지별로 루프 돌며 데이터 요청 시작
    for idx, apt in enumerate(target_apts):
        seq = apt.get("seq")
        name = apt.get("아파트명", seq)
        
        print(f"[{idx+1}/{len(target_apts)}] '{name}' (seq: {seq}) 수집 진행 중...")
        
        # ----------------------------------------------------
        # [과정 1] 단지 기본 정보 조회 (data_apt_dong.jsp)
        # ----------------------------------------------------
        info_params = {"apt": seq}
        try:
            info_response = requests.get(INFO_URL, headers=headers_info, params=info_params, timeout=10)
        
            if info_response.status_code != 200:
                print(f"1단계 단지정보 조회 실패 (Status: {info_response.status_code})")
                apt["status"] = "FAILED"
                continue
        except Exception as e:
            print(f"1단계 요청 중 오류 발생: {e}")
            apt["status"] = "FAILED"
            continue
            
        # 연속 요청 사이 지연 시간 적용
        time.sleep(0.5)

        # ----------------------------------------------------
        # [과정 2] 실제 매물 목록 조회 (data_sale_of_apt_nomal.aspx)
        # ----------------------------------------------------

        sale_payload = {
            "asil_bldcode": str(seq),
            "focus_bldcode": str(seq),
            "total": "20"
        }
                   
        try:
            sale_response = requests.post(SALE_URL, headers=headers_sale, data=sale_payload, timeout=10)
            
            if sale_response.status_code == 200:
                try:
                    response_json = sale_response.json()
                except ValueError:
                    print("JSON 파싱 에러: 매물 응답 본문이 JSON 형식이 아닙니다.")
                    apt["status"] = "FAILED"
                    continue

                data_list = response_json.get("list_result", [])
                
                # 데이터가 정상적인 리스트 구조인지 확인 후 수집 처리
                if isinstance(data_list, list) and len(data_list) > 0:
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    
                    for item in data_list:
                        if not isinstance(item, dict):
                            continue
                            
                        
                        forsale_info = {
                            "일련번호": seq,                       
                            "아파트명": name,
                            "수집시각": current_time,
                            "등록일자" : item.get("SVC_DATE_STRT",""),
                            "번호": item.get("MM_IDX", ""),
                            "매물ID": item.get("mm_uid", ""),
                            "거래타입": item.get("DEALTYPE_NM", ""),
                            "매물타입": item.get("SUB_RLSTTYPE_NM", ""),
                            "가격": item.get("DEAL_AMT", ""),                       
                            "동" : item.get("FLR_DP_MTHD_CD",""),
                            "층": item.get("CORES_FLR_CNT_NM", ""),                       
                            "면적": item.get("CTRT_SPC", ""),
                            "설명": item.get("FETR_DESC", ""),
                            "부동산중개소": item.get("BRKG_NM", ""),
                            "중개소연락처": item.get("TEL_ADD", "")
                        }
                        all_forsales.append(forsale_info)
                        
                    print(f"수집 성공: {len(data_list)}개의 매물 등록 완료.")
                    apt["status"] = "DONE"
                else:
                    print("등록된 실제 매물 데이터가 없거나 배열이 비어있습니다.")
                    apt["status"] = "DONE"
            else:
                print(f"2단계 매물 목록 조회 실패 (Status: {sale_response.status_code})")
                apt["status"] = "FAILED"
                
        except Exception as e:
            print(f"2단계 요청 중 오류 발생: {e}")
            apt["status"] = "FAILED"
        
        # [조건 4] 요청간 0.5초 이상 지연 규칙 준수 (0.6초)
        time.sleep(0.6)

    return all_forsales, apts_data


In [37]:
print(all_forsales)

([{'일련번호': '888', '아파트명': '리버탑', '수집시각': '2026-08-26 11:09:06', '등록일자': '2026-08-06', '번호': '1', '매물ID': '40831107', '거래타입': '매매', '매물타입': '아파트', '가격': '185,000', '동': '2', '층': '중', '면적': '105.80', '설명': '남향 한강뷰 특올수리 시스템에어컨', '부동산중개소': '주식회사센트로부동산중개법인', '중개소연락처': '010-3413-6886'}, {'일련번호': '2500106868', '아파트명': '몬트레아한남', '수집시각': '2026-08-26 11:09:08', '등록일자': '2026-08-24', '번호': '1', '매물ID': '41399820', '거래타입': '월세', '매물타입': '오피스텔', '가격': '0', '동': '1', '층': '3', '면적': '55.17', '설명': '몬트레아모든타입 최다호실보유 1.5룸 씨티대로변뷰 금액조율 입주협의', '부동산중개소': '반포트리니원하강남공인중개사사무소', '중개소연락처': '010-7108-8310'}, {'일련번호': '2500106868', '아파트명': '몬트레아한남', '수집시각': '2026-08-26 11:09:08', '등록일자': '2026-08-24', '번호': '2', '매물ID': '41399823', '거래타입': '매매', '매물타입': '오피스텔', '가격': '160,000', '동': '1', '층': '6', '면적': '103.59', '설명': '몬트레아모든타입 최다호실보유 1.5룸 씨티대로변뷰 급매 양창인기구조', '부동산중개소': '반포트리니원하강남공인중개사사무소', '중개소연락처': '010-7108-8310'}, {'일련번호': '2500107186', '아파트명': '브라이튼한남', '수집시각': '2026-08-26 11:09:09', '등록일자': '2026-08-25', '번

In [8]:
if __name__ == "__main__":
    all_forsales, apts_data = collect_forsale()

if all_forsales:
    forsales_file = "forsales.csv"
    first_sale = all_forsales[0]
    if isinstance(first_sale, dict):
        fieldnames = list(first_sale.keys())
    try:
        with open(forsales_file, mode="w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_forsales)
        print(f"[매물 저장 완료] 총 {len(all_forsales)}건의 매물이 '{forsales_file}'에 저장되었습니다.")
    except Exception as e:
        print(f"\n매물 파일 저장 중 오류 발생: {e}")
else:
    print("\n최종 수집된 매물 데이터가 없어 CSV 파일을 생성하지 않았습니다.")
        
    # 4. 변경된 status(DONE, FAILED)를 기존 apts.csv에 업데이트하여 저장
if apts_data:
    csv_file = "apts.csv"
    apts_fieldnames = list(apts_data[0].keys())
    try:
        with open(csv_file, mode="w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=apts_fieldnames)
            writer.writeheader()
            writer.writerows(apts_data)
        print(f"🔄 '{csv_file}'의 수집 상태(status) 최신화가 완료되었습니다.")
    except Exception as e:
        print(f"아파트 상태 파일 업데이트 실패: {e}")

총 20개 중 매물이 존재하는 6개의 아파트를 대상으로 수집을 진행합니다.

[1/6] '리버탑' (seq: 888) 수집 진행 중...
수집 성공: 1개의 매물 등록 완료.
[2/6] '몬트레아한남' (seq: 2500106868) 수집 진행 중...
수집 성공: 2개의 매물 등록 완료.
[3/6] '브라이튼한남' (seq: 2500107186) 수집 진행 중...
수집 성공: 6개의 매물 등록 완료.
[4/6] '한남더힐' (seq: 20135016) 수집 진행 중...
수집 성공: 1개의 매물 등록 완료.
[5/6] '한남리첸시아' (seq: 248315) 수집 진행 중...
수집 성공: 1개의 매물 등록 완료.
[6/6] '한남아이파크' (seq: 20348386) 수집 진행 중...
수집 성공: 3개의 매물 등록 완료.
[매물 저장 완료] 총 14건의 매물이 'forsales.csv'에 저장되었습니다.
🔄 'apts.csv'의 수집 상태(status) 최신화가 완료되었습니다.


## 요구사항 2. 재시도 & 로깅 & 실패 큐 처리 로직 적용

재시도 — 지수 백오프. 재시도 대상은 Timeout·ConnectionError·429·5xx로 한정하고, 404·400은 즉시 포기할 것

로깅 — print 대신 logging. 파일과 화면에 동시 출력, INFO/WARNING/ERROR 구분

실패 큐 — 실패한 seq을 오류 유형과 함께 failed_apt.csv, failed_forsale.csv 각각 분리하여 저장

통계 — 종료 시 성공 / 실패 / 건너뜀 건수를 한 줄로 출력

검증 — 큐의 URL 중 일부를 존재하지 않는 주소로 바꿔 넣고, 크롤러가 죽지 않고 끝까지 도는지 확인할 것

In [1]:
# 재시도

from requests.adapters import HTTPAdapter
import requests
from urllib3.util import Retry

def make_resilient_session():
    retry = Retry(
        total=4,  
        backoff_factor=1,  # 대기 시간 지수 증가 (1초 -> 2초 -> 4초 -> 8초)
        status_forcelist=[429, 500, 502, 503, 504],  # 이 에러들만 재시도 수행
        allowed_methods=["GET", "POST"],
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session

In [ ]:
# 로깅

import logging


logging.basicConfig(
    level=logging.INFO,  
    format="%(asctime)s [%(levelname)s] %(message)s",  # 시간 및 레벨 포맷
    handlers=[
        logging.FileHandler("crawler.log", encoding="utf-8"),  # 파일 저장
        logging.StreamHandler(),  # 터미널 화면 출력
    ],
)
logger = logging.getLogger(__name__)


def collect_forsale():
    INFO_URL = "https://asil.kr/app/data/data_apt_dong.jsp?"
    SALE_URL = "https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx"

    apts_file = "apts.csv"

    # 1. 기존 apts.csv 파일 읽기 및 대상 필터링
    apts_data = []
    try:
        with open(apts_file, mode="r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            apts_data = list(reader)
    except FileNotFoundError:
        logger.error(
            f"'{apts_file}' 파일이 존재하지 않습니다. 1단계를 먼저 실행해 주세요."
        )
        return

    target_apts = []
    for apt in apts_data:
        offer_val = apt.get("매물수", "0").strip()
        if not offer_val or offer_val == "0" or offer_val == "":
            apt["status"] = "DONE"
        if offer_val and offer_val != "0" and apt.get("status") == "PENDING":
            target_apts.append(apt)

    logger.info(
        f"총 {len(apts_data)}개 중 매물이 존재하는 {len(target_apts)}개의 아파트를 대상으로 수집을 진행합니다."
    )

    if not target_apts:
        logger.info("수집할 대상(PENDING 상태 및 매물 보유)이 없습니다.")
        return

    all_forsales = []

    # 2. 필터링된 아파트 단지별로 루프 돌며 데이터 요청 시작
    for idx, apt in enumerate(target_apts):
        seq = apt.get("seq")
        name = apt.get("아파트명", seq)

        logger.info(
            f"[{idx+1}/{len(target_apts)}] '{name}' (seq: {seq}) 매물 상세 수집 시작..."
        )

        # ----------------------------------------------------
        # [과정 1] 단지 기본 정보 조회 (data_apt_dong.jsp)
        # ----------------------------------------------------
        info_params = {"apt": seq}
        try:
            info_response = requests.get(
                INFO_URL, headers=headers_info, params=info_params, timeout=10
            )

            if info_response.status_code != 200:
                logger.warning(
                    f"1단계 단지정보 조회 실패 (Status: {info_response.status_code}) | 단지: {name}"
                )
                apt["status"] = "FAILED"
                continue
        except Exception as e:
            logger.error(
                f"1단계 네트워크 요청 중 오류 발생: {e} | 단지: {name}"
            )
            apt["status"] = "FAILED"
            continue

        # 연속 요청 사이 지연 시간 적용
        time.sleep(0.5)

        # ----------------------------------------------------
        # [과정 2] 실제 매물 목록 조회 (data_sale_of_apt_nomal.aspx)
        # ----------------------------------------------------
        sale_payload = {
            "asil_bldcode": str(seq),
            "focus_bldcode": str(seq),
            "total": "20",
        }

        try:
            sale_response = requests.post(
                SALE_URL, headers=headers_sale, data=sale_payload, timeout=10
            )

            if sale_response.status_code == 200:
                try:
                    response_json = sale_response.json()
                except ValueError:
                    logger.error(
                        f"JSON 파싱 에러: 매물 응답 본문이 JSON 형식이 아닙니다. | 단지: {name}"
                    )
                    apt["status"] = "FAILED"
                    continue

                data_list = response_json.get("list_result", [])

                
                if isinstance(data_list, list) and len(data_list) > 0:
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    for item in data_list:
                        if not isinstance(item, dict):
                            continue

                        forsale_info = {
                            "일련번호": seq,
                            "아파트명": name,
                            "수집시각": current_time,
                            "등록일자": item.get("SVC_DATE_STRT", ""),
                            "번호": item.get("MM_IDX", ""),
                            "매물ID": item.get("mm_uid", ""),
                            "거래타입": item.get("DEALTYPE_NM", ""),
                            "매물타입": item.get("SUB_RLSTTYPE_NM", ""),
                            "가격": item.get("DEAL_AMT", ""),
                            "동": item.get("FLR_DP_MTHD_CD", ""),
                            "층": item.get("CORES_FLR_CNT_NM", ""),
                            "면적": item.get("CTRT_SPC", ""),
                            "설명": item.get("FETR_DESC", ""),
                            "부동산중개소": item.get("BRKG_NM", ""),
                            "중개소연락처": item.get("TEL_ADD", ""),
                        }
                        all_forsales.append(forsale_info)

                    logger.info(
                        f"수집 성공: {len(data_list)}개의 매물 등록 완료."
                    )
                    apt["status"] = "DONE"
                else:
                    logger.warning(
                        f"등록된 실제 매물 데이터가 없거나 배열이 비어있습니다. | 단지: {name}"
                    )
                    apt["status"] = "DONE"
            else:
                logger.warning(
                    f"2단계 매물 목록 조회 실패 (Status: {sale_response.status_code}) | 단지: {name}"
                )
                apt["status"] = "FAILED"

        except Exception as e:
            logger.error(
                f" 2단계 네트워크 요청 중 오류 발생: {e} | 단지: {name}"
            )
            apt["status"] = "FAILED"

        # [조건 4] 요청간 0.5초 이상 지연 규칙 준수 (0.6초)
        time.sleep(0.6)

    logger.info("🏁 모든 아파트 대상 수집 루프가 종료되었습니다.")
    return all_forsales, apts_data

In [ ]:
# 실패 큐  - Failed가 없어 미시행
try:
    sale_response = session.post(
        SALE_URL, headers=headers_sale, data=sale_payload, timeout=10
    )
    sale_response.raise_for_status() 
except Exception as e:
    logger.error(f"단지 {seq} 수집 중 오류 발생: {e}") 

    # 실패 큐 분리 기록
    with open("failed_forsale.csv", mode="a", encoding="utf-8") as f:
        f.write(f"{seq},{type(e).__name__}\n")

    apt["status"] = "FAILED"
    continue

In [ ]:
# 통계

from tqdm import tqdm 

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("crawler_forsale.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)


def collect_forsale():
    INFO_URL = "https://asil.kr/app/data/data_apt_dong.jsp?"
    SALE_URL = "https://realty.asil.kr/api_asil/data_sale_of_apt_nomal.aspx"

    apts_file = "apts.csv"

    # 1. 기존 apts.csv 파일 읽기
    apts_data = []
    try:
        with open(apts_file, mode="r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            apts_data = list(reader)
    except FileNotFoundError:
        logger.error(
            f"'{apts_file}' 파일이 존재하지 않습니다. 1단계를 먼저 실행해 주세요."
        )
        return

    stat = {"ok": 0, "fail": 0, "skip": 0}
    target_apts = []

    for apt in apts_data:
        offer_val = apt.get("매물수", "0").strip()

        if not offer_val or offer_val == "0" or offer_val == "":
            apt["status"] = "DONE"
            stat["skip"] += 1
        elif apt.get("status") != "PENDING":
            stat["skip"] += 1
        else:
            target_apts.append(apt)

    logger.info(
        f"총 {len(apts_data)}개 단지 중 수집 대상: {len(target_apts)}개 / 건너뜀: {stat['skip']}개"
    )

    if not target_apts:
        logger.info("현재 수집할 대상(PENDING 상태 및 매물 보유)이 없습니다.")
        logger.info(
            "[수집 완료] 성공:%(ok)d · 실패:%(fail)d · 건너뜀:%(skip)d", stat
        )
        return [], apts_data

    all_forsales = []

    for idx, apt in enumerate(
        tqdm(target_apts, desc="아실 매물 상세 수집 진행률")
    ):
        seq = apt.get("seq")
        name = apt.get("아파트명", seq)

        # ----------------------------------------------------
        # [과정 1] 단지 기본 정보 조회
        # ----------------------------------------------------
        info_params = {"apt": seq}
        try:
            info_response = requests.get(
                INFO_URL, headers=headers_info, params=info_params, timeout=10
            )
            if info_response.status_code != 200:
                apt["status"] = "FAILED"
                stat["fail"] += 1
                continue
        except Exception as e:
            apt["status"] = "FAILED"
            stat["fail"] += 1
            continue

        time.sleep(0.5)

        # ----------------------------------------------------
        # [과정 2] 실제 매물 목록 조회 (POST)
        # ----------------------------------------------------
        sale_payload = {
            "asil_bldcode": str(seq),
            "focus_bldcode": str(seq),
            "total": "20",
        }

        try:
            sale_response = requests.post(
                SALE_URL, headers=headers_sale, data=sale_payload, timeout=10
            )

            if sale_response.status_code == 200:
                try:
                    response_json = sale_response.json()
                except ValueError:
                    apt["status"] = "FAILED"
                    stat["fail"] += 1
                    continue

                data_list = response_json.get("list_result", [])

                if isinstance(data_list, list) and len(data_list) > 0:
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    for item in data_list:
                        if not isinstance(item, dict):
                            continue

                        forsale_info = {
                            "일련번호": seq,
                            "아파트명": name,
                            "수집시각": current_time,
                            "등록일자": item.get("SVC_DATE_STRT", ""),
                            "번호": item.get("MM_IDX", ""),
                            "매물ID": item.get("mm_uid", ""),
                            "거래타입": item.get("DEALTYPE_NM", ""),
                            "매물타입": item.get("SUB_RLSTTYPE_NM", ""),
                            "가격": item.get("DEAL_AMT", ""),
                            "동": item.get("FLR_DP_MTHD_CD", ""),
                            "층": item.get("CORES_FLR_CNT_NM", ""),
                            "면적": item.get("CTRT_SPC", ""),
                            "설명": item.get("FETR_DESC", ""),
                            "부동산중개소": item.get("BRKG_NM", ""),
                            "중개소연락처": item.get("TEL_ADD", ""),
                        }
                        all_forsales.append(forsale_info)

                    apt["status"] = "DONE"
                    stat["ok"] += 1
                else:
                    apt["status"] = "DONE"
                    stat["ok"] += 1 
            else:
                apt["status"] = "FAILED"
                stat["fail"] += 1

        except Exception as e:
            apt["status"] = "FAILED"
            stat["fail"] += 1

        time.sleep(0.6)

    logger.info(
        "[수집 완료] 성공:%(ok)d · 실패:%(fail)d · 건너뜀:%(skip)d", stat
    )

    return all_forsales, apts_data

In [ ]:
# 검증

for idx, apt in enumerate(target_apts):
    
    try:
        seq = apt.get("seq")
        name = apt.get("아파트명", seq)

        # ----------------------------------------------------
        # [과정 1] 단지 기본 정보 조회 (통신)
        # ----------------------------------------------------
        info_response = requests.get(INFO_URL, headers=headers_info, params={"apt": seq}, timeout=10)
        info_response.raise_for_status()
        
        # ----------------------------------------------------
        # [과정 2] 실제 매물 목록 조회 (통신 및 파싱)
        # ----------------------------------------------------
        sale_response = requests.post(SALE_URL, headers=headers_sale, data={"apt": str(seq)}, timeout=10)
        sale_response.raise_for_status()
        
        apt["status"] = "DONE"
        stat["ok"] += 1

    except Exception as general_error:
        logger.error(f"검증 가드에 걸린 오류 [단지: {name} (seq: {seq})]: {general_error}")
        
        apt["status"] = "FAILED"
        stat["fail"] += 1
        continue  

    finally:
        time.sleep(0.6)